# Exercise 2 — Guided transfer learning with CIFAR-10

**Computer Vision for Industrial Systems**  
**Lecture context:** Machine learning foundations for industrial vision

This notebook gives you a guided first transfer-learning pipeline in PyTorch.

You will:

1. load CIFAR-10 through TorchVision,
2. define preprocessing and augmentation,
3. inspect image tensors and labels,
4. adapt pretrained CNN backbones,
5. train three lightweight transfer-learning baselines,
6. compare validation metrics and learning curves,
7. inspect errors.

CIFAR-10 is not an industrial dataset. It is used first because it is small, standard, directly loadable, and useful for learning the workflow before moving to an industrial dataset.

Difficulty markers:

- 🟢 basic
- 🟡 intermediate
- 🔴 advanced / optional

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(repo_root / "src"))

from cvis_ml.config import DatasetConfig, ModelConfig, TrainConfig, ExperimentConfig
from cvis_ml.data import CIFAR10DataModule
from cvis_ml.models import TransferModelFactory, count_parameters, describe_trainable_parameters
from cvis_ml.engine import Trainer
from cvis_ml.visualization import show_batch, plot_history, show_confusion_matrix

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

## Part 0 — CNN intuition in one cell

A convolutional layer applies small filters to an image tensor.

For an RGB image, the tensor shape is usually:

$$
C \times H \times W
$$

where:

| Symbol | Meaning |
|---|---|
| `C` | number of channels |
| `H` | image height |
| `W` | image width |

A convolutional layer maps input channels to output channels:

$$
\text{input tensor} \rightarrow \text{feature maps}
$$

In a pretrained CNN, early filters often respond to edges, colors, and simple texture patterns. Later layers combine these into more task-specific features. In transfer learning, we reuse this feature extractor and adapt the final classification head.

In [ ]:
# Inspect the shape of a convolutional layer.
conv = torch.nn.Conv2d(in_channels=3, out_channels=8, kernel_size=3, padding=1)
x = torch.randn(4, 3, 64, 64)  # batch of 4 RGB images
z = conv(x)

print("input shape :", tuple(x.shape))
print("output shape:", tuple(z.shape))
print("weight shape:", tuple(conv.weight.shape))

## Part 1 — Dataset and preprocessing

Preprocessing is part of the model contract.

For pretrained CNNs, we usually need:

1. resize to a common image size,
2. convert image to tensor,
3. normalize with ImageNet mean and standard deviation,
4. use augmentation only if it is plausible for the task.

In this first notebook, CIFAR-10 images are resized to `96 x 96` so the pretrained CNNs can process them comfortably.

## Task 1.1 — Configure the dataset 🟢

Fill in a dataset configuration.

Keep the subset small enough for a 90-minute exercise.

In [ ]:
# TODO:
# 1. Choose image_size = 96.
# 2. Choose batch_size = 32.
# 3. Choose max_train_samples around 1500 or 2000.
# 4. Choose max_val_samples around 400 or 500.

cifar_cfg = DatasetConfig(
    name="cifar10",
    data_root=str(repo_root / "data_cache"),
    image_size=...,          # TODO
    batch_size=...,          # TODO
    max_train_samples=...,   # TODO
    max_val_samples=...,     # TODO
    num_workers=2,
    seed=42,
)

cifar_cfg

## Task 1.2 — Load data 🟢

The `CIFAR10DataModule` is a small object-oriented wrapper around TorchVision.

It handles:

- dataset download,
- preprocessing,
- augmentation,
- balanced subset creation,
- DataLoader creation.

Call `setup()` to create the training and validation loaders.

In [ ]:
# TODO:
# 1. Create a CIFAR10DataModule from cifar_cfg.
# 2. Call setup() to obtain a DataBundle.

cifar_dm = ...
cifar_data = ...

print("Classes:", cifar_data.class_names)
print("Number of classes:", cifar_data.num_classes)
print("Train batches:", len(cifar_data.train_loader))
print("Validation batches:", len(cifar_data.val_loader))

show_batch(cifar_data.train_loader, cifar_data.class_names, n=8)

## Part 2 — Transfer-learning baselines

We compare three lightweight CNN baselines:

1. **ResNet18 frozen feature extractor**  
   Backbone frozen, only the new classification head is trained.  
   Paper: He et al., *Deep Residual Learning for Image Recognition*, CVPR 2016.

2. **ResNet18 partial fine-tuning**  
   Backbone mostly frozen, but the last residual block and classification head are trained.

3. **MobileNetV3-Small frozen feature extractor**  
   Efficient mobile-oriented CNN, trained head only.  
   Paper: Howard et al., *Searching for MobileNetV3*, ICCV 2019.

The goal is not to get perfect accuracy. The goal is to understand the workflow and trade-offs.

## Task 2.1 — Create one transfer-learning model 🟢

Create a pretrained ResNet18 with a new classification head.

Strategy:

- `frozen`: train only the head,
- `partial`: train later layers and head,
- `full`: train everything.

In [ ]:
# TODO:
# Create a ResNet18 frozen feature-extractor model.

model = TransferModelFactory.create(
    architecture=...,      # TODO: "resnet18"
    num_classes=...,       # TODO: cifar_data.num_classes
    strategy=...,          # TODO: "frozen"
    pretrained=True,
)

total_params, trainable_params = count_parameters(model)
print("Total parameters    :", total_params)
print("Trainable parameters:", trainable_params)
print("Trainable tensors:")
for row in describe_trainable_parameters(model):
    print(row)

## Task 2.2 — Train the first baseline 🟢

Now train the frozen ResNet18 baseline for one epoch.

This is intentionally short. On CPU it may still take some time.

In [ ]:
# TODO:
# 1. Create a Trainer.
# 2. Train the model using fit().

trainer = Trainer(
    model=model,
    device="auto",
    learning_rate=1e-3,
    weight_decay=1e-4,
)

result_resnet_frozen = trainer.fit(
    train_loader=cifar_data.train_loader,
    val_loader=cifar_data.val_loader,
    epochs=1,
    max_batches_per_epoch=20,
    name="resnet18_frozen",
)

plot_history(result_resnet_frozen.history, title="ResNet18 frozen")
show_confusion_matrix(result_resnet_frozen.y_true, result_resnet_frozen.y_pred, cifar_data.class_names, title="ResNet18 frozen")

## Task 2.3 — Run a small baseline ladder 🟡

Now define three experiment configurations.

You should compare:

1. `resnet18` with `frozen`,
2. `resnet18` with `partial`,
3. `mobilenet_v3_small` with `frozen`.

Keep training short: `epochs=1`, `max_batches_per_epoch=20`.

In [ ]:
# TODO:
# Complete the experiment list.

experiments = [
    {"name": "resnet18_frozen", "architecture": ..., "strategy": ...},
    {"name": "resnet18_partial", "architecture": ..., "strategy": ...},
    {"name": "mobilenet_v3_small_frozen", "architecture": ..., "strategy": ...},
]

all_results = []

for exp in experiments:
    print("\nRunning:", exp)
    model_i = TransferModelFactory.create(
        architecture=exp["architecture"],
        num_classes=cifar_data.num_classes,
        strategy=exp["strategy"],
        pretrained=True,
    )
    total, trainable = count_parameters(model_i)
    print(f"Parameters: total={total:,}, trainable={trainable:,}")

    trainer_i = Trainer(
        model=model_i,
        device="auto",
        learning_rate=1e-3 if exp["strategy"] == "frozen" else 1e-4,
        weight_decay=1e-4,
    )
    result_i = trainer_i.fit(
        cifar_data.train_loader,
        cifar_data.val_loader,
        epochs=1,
        max_batches_per_epoch=20,
        name=exp["name"],
    )
    all_results.append({
        "experiment": exp["name"],
        "architecture": exp["architecture"],
        "strategy": exp["strategy"],
        "total_params": total,
        "trainable_params": trainable,
        "val_accuracy": result_i.metrics["accuracy"],
        "val_macro_f1": result_i.metrics["macro_f1"],
        "runtime_s": result_i.metrics["runtime_s"],
    })

pd.DataFrame(all_results)

## Task 2.4 — Interpretation 🟡

Answer in 5–8 sentences:

1. Which baseline performed best?
2. Which baseline had the fewest trainable parameters?
3. Did partial fine-tuning clearly improve over a frozen backbone?
4. Why might the answer depend on dataset size and learning rate?
5. Which result would you trust most as a first industrial baseline, and why?

In [ ]:
answer_2_4 = """


"""
print(answer_2_4)

## Advanced optional task 🔴

Try one of the following:

- increase `max_batches_per_epoch`,
- increase `epochs` to 2,
- try `efficientnet_b0`,
- change augmentation,
- compare results with and without augmentation.

Remember: in industry, augmentations must represent plausible operational variation.